In [1]:
!pip install ratinabox
!pip install pandas

  Using cached ratinabox-1.15.3-py3-none-any.whl.metadata (34 kB)


  Using cached shapely-2.1.2-cp314-cp314-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (6.8 kB)


Using cached ratinabox-1.15.3-py3-none-any.whl (2.7 MB)
Using cached shapely-2.1.2-cp314-cp314-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (3.1 MB)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/2 [shapely]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [ratinabox]


In [ ]:
import os
import sys
from pathlib import Path

# Match notebooks 01-05: resolve paths relative to the repo root regardless of
# whether this runs interactively (cwd == notebooks/) or headlessly via nbconvert
# (cwd also defaults to notebooks/), so "data/..." always lands in the repo root.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import multiprocessing as mp
# this environment's default start method (forkserver) can't see functions/globals defined
# interactively in the notebook; "fork" inherits the notebook process's memory directly.
# Must be set once, before any Pool is created.
mp.set_start_method("fork", force=True)
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d
from ratinabox.Environment import Environment


In [3]:
# 2.2m x 2.2m square box; motion generated using the Raudies & Hasselmo (2012) rat motion
# model with Table 1 (Supplementary Methods) parameters from Banino et al. 2018 (Nature)
# https://www.nature.com/articles/s41586-018-0102-6
Env = Environment(params={'scale': 2.2})

T = 15                        # trajectory duration in seconds (Table 1)
dt = 0.02                     # Δt: simulation-step time increment (s)
L = Env.scale                 # 2.2, environment width/height (m)
d = 0.03                       # perimeter distance to walls (m)
sigma_v = 0.13                # σ(v): forward velocity Rayleigh scale (m/s)
sigma_phi = np.deg2rad(330)   # σ(φ): rotational velocity Gaussian std (rad/s); μ(φ)=0
rho_RH = 0.25                 # velocity reduction factor near walls
# Δ_RH (90°: redirect heading parallel to the nearby wall) is implemented geometrically
# in _wall_redirect() below rather than as a single scalar

n_steps_per_trajectory = 100    # timesteps stored per trajectory (Table 1)
n_fine_steps = round(T / dt)    # 750 fine simulation steps per trajectory
t_target = np.linspace(0, T, n_steps_per_trajectory, endpoint=False)  # 0, 0.15, ..., 14.85s

In [4]:
def _wall_zone(ang, pos, min_gap, max_gap):
    """True if heading `ang` at `pos` is aimed toward a wall within the d-perimeter."""
    if (0 <= ang <= np.pi / 2) and np.any(pos > max_gap):
        return True
    elif (np.pi / 2 <= ang <= np.pi) and (pos[0] < min_gap or pos[1] > max_gap):
        return True
    elif (-np.pi <= ang <= -np.pi / 2) and np.any(pos < min_gap):
        return True
    elif (-np.pi / 2 <= ang <= 0) and (pos[0] > max_gap or pos[1] < min_gap):
        return True
    return False


def _wall_redirect(ang, pos, min_gap, max_gap):
    """Angle change redirecting heading `ang` to run parallel to the nearby wall (Δ_RH)."""
    rot = 0.0
    if 0 <= ang <= np.pi / 2:
        if pos[1] > max_gap: rot = -ang
        elif pos[0] > max_gap: rot = np.pi / 2 - ang
    elif np.pi / 2 <= ang <= np.pi:
        if pos[1] > max_gap: rot = np.pi - ang
        elif pos[0] < min_gap: rot = np.pi / 2 - ang
    elif -np.pi <= ang <= -np.pi / 2:
        if pos[1] < min_gap: rot = -np.pi - ang
        elif pos[0] < min_gap: rot = -(ang + np.pi / 2)
    else:
        if pos[1] < min_gap: rot = -ang
        elif pos[0] > max_gap: rot = -np.pi / 2 - ang
    return rot


def simulate_rh_trajectory(L, d, sigma_v, sigma_phi, rho_RH, dt, n_steps):
    """Raudies & Hasselmo (2012) rat motion model. Returns (n_steps+1)-length arrays:
    position (x,y), heading angle, linear speed, and realized angular velocity (rad/s)."""
    min_gap, max_gap = d, L - d
    pos = np.zeros((n_steps + 1, 2))
    ang = np.zeros(n_steps + 1)
    vel = np.zeros(n_steps + 1)
    rot_vel = np.zeros(n_steps + 1)

    pos[0] = np.random.uniform(0, L, size=2)
    ang[0] = np.random.uniform(-np.pi, np.pi)
    prev_vel = 0.0

    for t in range(1, n_steps + 1):
        cur_pos, cur_ang = pos[t - 1], ang[t - 1]
        if _wall_zone(cur_ang, cur_pos, min_gap, max_gap):
            rot_sample = np.random.normal(0, sigma_phi)
            dAngle = _wall_redirect(cur_ang, cur_pos, min_gap, max_gap) + rot_sample * dt
            v = prev_vel * (1 - rho_RH)
        else:
            v = np.random.rayleigh(sigma_v)
            rot_sample = np.random.normal(0, sigma_phi)
            dAngle = rot_sample * dt

        new_pos = cur_pos + np.array([np.cos(cur_ang), np.sin(cur_ang)]) * v * dt
        new_ang = cur_ang + dAngle
        if abs(new_ang) >= np.pi:
            new_ang = -np.sign(new_ang) * (np.pi - (abs(new_ang) - np.pi))

        pos[t], ang[t], vel[t], rot_vel[t] = new_pos, new_ang, v, dAngle / dt
        prev_vel = v

    return pos, ang, vel, rot_vel

In [5]:
# file-batch config: matches the paper's 100-files x 10,000-trajectories structure (1,000,000 total).
n_files = 100
trajectories_per_file = 10_000
output_dir = "data/square_room_100steps_2.2m_1000000"
os.makedirs(output_dir, exist_ok=True)


def generate_file(file_idx):
    """Generates one file's worth of trajectories and writes it straight to disk, so a
    worker process never needs to hold more than one file's rows in memory at a time."""
    np.random.seed(file_idx)  # distinct RNG stream per file/worker (fork copies parent RNG state)
    rows = []
    for local_traj_id in range(trajectories_per_file):
        pos_fine, ang_fine, vel_fine, rot_vel_fine = simulate_rh_trajectory(
            L, d, sigma_v, sigma_phi, rho_RH, dt, n_fine_steps)

        t_fine = np.linspace(0, T, n_fine_steps + 1)
        hd_fine = np.column_stack([np.cos(ang_fine), np.sin(ang_fine)])
        vel_fine_xy = vel_fine[:, None] * hd_fine
        dist_fine = np.concatenate([[0.0], np.cumsum(vel_fine[1:] * dt)])

        pos_t = interp1d(t_fine, pos_fine, axis=0, kind="cubic")(t_target)
        vel_t = interp1d(t_fine, vel_fine_xy, axis=0, kind="linear")(t_target)
        rot_vel_t = interp1d(t_fine, rot_vel_fine, kind="linear")(t_target)
        hd_t = interp1d(t_fine, hd_fine, axis=0, kind="cubic")(t_target)
        hd_t /= np.linalg.norm(hd_t, axis=1, keepdims=True)
        dist_t = interp1d(t_fine, dist_fine, kind="linear")(t_target)

        global_traj_id = file_idx * trajectories_per_file + local_traj_id
        for step in range(n_steps_per_trajectory):
            rows.append({
                "trajectory_id": global_traj_id,
                "step": step,
                "t": t_target[step],
                "pos_x": pos_t[step, 0],
                "pos_y": pos_t[step, 1],
                "vel_x": vel_t[step, 0],
                "vel_y": vel_t[step, 1],
                "rot_vel": rot_vel_t[step],
                "head_direction_x": hd_t[step, 0],
                "head_direction_y": hd_t[step, 1],
                "distance_travelled": dist_t[step],
            })

    path = os.path.join(output_dir, f"{file_idx:04d}-of-{n_files - 1:04d}.csv")
    pd.DataFrame(rows).to_csv(path, index=False)
    return path

In [6]:
n_workers = os.cpu_count()
with mp.Pool(n_workers) as pool:
    paths = pool.map(generate_file, range(n_files))

print(f"Wrote {len(paths)} files to {output_dir}/ using {n_workers} workers")

Wrote 100 files to data/square_room_100steps_2.2m_1000000/ using 16 workers


In [7]:
# sanity check: load one generated file back and verify structure
sample_df = pd.read_csv(paths[0])
print(sample_df.shape)
print(sample_df.groupby("trajectory_id").size().describe())
sample_df.head()

(1000000, 11)
count    10000.0
mean       100.0
std          0.0
min        100.0
25%        100.0
50%        100.0
75%        100.0
max        100.0
dtype: float64


,trajectory_id,step,t,pos_x,pos_y,vel_x,vel_y,rot_vel,head_direction_x,head_direction_y,distance_travelled
0,0,0,0.00,1.207390,1.573417,0.000000,0.000000,0.000000,0.798690,0.601743,0.000000
1,0,1,0.15,1.224792,1.590796,0.117619,0.171542,2.239147,0.585509,0.810666,0.025044
2,0,2,0.30,1.238182,1.610922,0.069489,0.166331,8.828175,0.385488,0.922713,0.048964
3,0,3,0.45,1.246330,1.632359,0.085514,0.128377,-5.557051,0.567020,0.823704,0.072003
4,0,4,0.60,1.261913,1.646583,0.277792,0.223573,2.467014,0.779034,0.626982,0.093304


In [8]:
# total rows across all files should match n_files x trajectories_per_file x n_steps_per_trajectory
total_rows = sum(len(pd.read_csv(p)) for p in paths)
expected_rows = n_files * trajectories_per_file * n_steps_per_trajectory
print(f"total rows: {total_rows}, expected: {expected_rows}, match: {total_rows == expected_rows}")

total rows: 100000000, expected: 100000000, match: True


In [9]:
# each file should cover a distinct, non-overlapping trajectory_id range, with no dropped trajectories
for p in paths:
    d_ = pd.read_csv(p)
    n_traj = d_["trajectory_id"].nunique()
    print(p, "trajectory_id range:", d_.trajectory_id.min(), "-", d_.trajectory_id.max(), f"({n_traj} trajectories)")

data/square_room_100steps_2.2m_1000000/0000-of-0099.csv trajectory_id range: 0 - 9999 (10000 trajectories)


data/square_room_100steps_2.2m_1000000/0001-of-0099.csv trajectory_id range: 10000 - 19999 (10000 trajectories)


data/square_room_100steps_2.2m_1000000/0002-of-0099.csv trajectory_id range: 20000 - 29999 (10000 trajectories)


data/square_room_100steps_2.2m_1000000/0003-of-0099.csv trajectory_id range: 30000 - 39999 (10000 trajectories)


data/square_room_100steps_2.2m_1000000/0004-of-0099.csv trajectory_id range: 40000 - 49999 (10000 trajectories)


...

data/square_room_100steps_2.2m_1000000/0095-of-0099.csv trajectory_id range: 950000 - 959999 (10000 trajectories)


data/square_room_100steps_2.2m_1000000/0096-of-0099.csv trajectory_id range: 960000 - 969999 (10000 trajectories)


data/square_room_100steps_2.2m_1000000/0097-of-0099.csv trajectory_id range: 970000 - 979999 (10000 trajectories)


data/square_room_100steps_2.2m_1000000/0098-of-0099.csv trajectory_id range: 980000 - 989999 (10000 trajectories)


data/square_room_100steps_2.2m_1000000/0099-of-0099.csv trajectory_id range: 990000 - 999999 (10000 trajectories)
